# DreamSnap generation worker (Colab GPU)

Keep this notebook running in a GPU session to process queued single-image and pack jobs. It polls the DreamSnap backend, generates with FLUX.1-dev, uploads images to the private S3 bucket through presigned URLs, and reports job status.

Before running: enable a GPU runtime, accept the FLUX.1-dev license on Hugging Face, and add Colab Secrets named `BACKEND_URL`, `WORKER_SECRET`, and `HF_TOKEN`. Use the same backend URL and worker secret configured on the backend. Do not put secrets in notebook cells.

In [ ]:
!pip -q install "diffusers>=0.32.0" transformers accelerate peft sentencepiece protobuf safetensors huggingface_hub requests

In [ ]:
import random
import tempfile
import time
from pathlib import Path

import requests
import torch
from diffusers import FluxPipeline
from google.colab import userdata
from huggingface_hub import login

BACKEND_URL = userdata.get("BACKEND_URL").rstrip("/")
WORKER_SECRET = userdata.get("WORKER_SECRET")
HF_TOKEN = userdata.get("HF_TOKEN")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime before starting the worker.")
if not BACKEND_URL.startswith(("https://", "http://localhost", "http://127.0.0.1")):
    raise ValueError("BACKEND_URL must use HTTPS except for a local development server.")

login(token=HF_TOKEN, add_to_git_credential=False)
HEADERS = {"x-worker-secret": WORKER_SECRET}
MODEL_ID = "black-forest-labs/FLUX.1-dev"
print("Loading FLUX.1-dev. CPU offload reduces VRAM requirements but can be slow on a free GPU.")
pipe = FluxPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    token=HF_TOKEN,
    low_cpu_mem_usage=True,
)
pipe.enable_model_cpu_offload()
print("Generation pipeline ready.")

In [ ]:
def generate_job(job):
    previous_lora = getattr(generate_job, "loaded_lora", None)
    if job.get("loraUrl") != previous_lora:
        if previous_lora:
            pipe.unload_lora_weights()
        if job.get("loraUrl"):
            response = requests.get(job["loraUrl"], timeout=120)
            response.raise_for_status()
            lora_path = Path(tempfile.gettempdir()) / f"dreamsnap-{job['id']}.safetensors"
            lora_path.write_bytes(response.content)
            pipe.load_lora_weights(str(lora_path))
        generate_job.loaded_lora = job.get("loraUrl")

    dimensions = {"1:1": (512, 512), "3:2": (768, 512), "2:3": (512, 768)}
    width, height = dimensions.get(job.get("ratio"), (768, 512))
    images = []
    for index in range(job["imageCount"]):
        result = pipe(
            prompt=f"{job['prompt']}. {job.get('style', 'Dreamy')} style.",
            width=width,
            height=height,
            num_inference_steps=24,
            guidance_scale=3.5,
            generator=torch.Generator("cpu").manual_seed(random.randrange(0, 2**31)),
        )
        image_path = Path(tempfile.gettempdir()) / f"dreamsnap-{job['id']}-{index}.png"
        result.images[0].save(image_path)
        images.append(image_path)
    return images

def process_generation_job(job):
    try:
        image_paths = generate_job(job)
        output_keys = []
        for index, image_path in enumerate(image_paths):
            ticket = requests.post(
                f"{BACKEND_URL}/worker/jobs/{job['id']}/upload-url",
                headers=HEADERS,
                json={"index": index, "contentType": "image/png"},
                timeout=30,
            )
            ticket.raise_for_status()
            upload = ticket.json()
            with image_path.open("rb") as image_file:
                uploaded = requests.put(upload["uploadUrl"], data=image_file, headers={"Content-Type": "image/png"}, timeout=180)
            uploaded.raise_for_status()
            output_keys.append(upload["key"])
            image_path.unlink(missing_ok=True)

        completed = requests.post(
            f"{BACKEND_URL}/worker/jobs/{job['id']}/complete",
            headers=HEADERS,
            json={"outputKeys": output_keys},
            timeout=30,
        )
        completed.raise_for_status()
        print(f"Completed {job['kind']} job {job['id']} ({len(output_keys)} image(s)).")
    except Exception as error:
        print(f"Generation job {job['id']} failed: {error}")
        requests.post(
            f"{BACKEND_URL}/worker/jobs/{job['id']}/fail",
            headers=HEADERS,
            json={"error": str(error)[:500]},
            timeout=30,
        ).raise_for_status()

In [ ]:
def run_generation_worker():
    print("Generation worker is polling. Interrupt this cell to stop it.")
    while True:
        try:
            response = requests.get(f"{BACKEND_URL}/worker/jobs", headers=HEADERS, timeout=30)
            response.raise_for_status()
            job = response.json().get("job")
            if job:
                process_generation_job(job)
            else:
                time.sleep(8)
        except KeyboardInterrupt:
            print("Generation worker stopped.")
            break
        except Exception as error:
            print(f"Worker polling error: {error}")
            time.sleep(15)

run_generation_worker()